In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import json
from datetime import datetime

print("="*70)
print("EXTRACT BEST OUTPUTS FROM ALL 5 MODELS")
print("="*70)
print(f"\nStarted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Define model files
model_files = {
    'Ensemble': 'solar_energy_forecast_2026.csv',
    'SARIMA': 'sarima_forecast_2026.csv',
    'Prophet': 'prophet_forecast_2026.csv',
    'LSTM': 'lstm_forecast_2026.csv',
    'XGBoost': 'xgboost_forecast_2026.csv'
}

# Check which files exist
available_models = {}
missing_models = []

for model, filename in model_files.items():
    if Path(filename).exists():
        available_models[model] = filename
        print(f"✓ {model:<15} -> {filename}")
    else:
        missing_models.append(model)
        print(f"✗ {model:<15} -> {filename} (NOT FOUND)")

if missing_models:
    print(f"\nWarning: {len(missing_models)} model(s) not found: {', '.join(missing_models)}")
    print("Please run the respective model notebooks first.")

print(f"\nLoading {len(available_models)} available model forecasts...")

In [ ]:
# Load all available forecasts
forecasts = {}
for model, filename in available_models.items():
    df = pd.read_csv(filename)
    df['Date'] = pd.to_datetime(df['Date'])
    forecasts[model] = df
    print(f"Loaded {model:<15}: {len(df)} days, {df.shape[1]} columns")

print(f"\nTotal models loaded: {len(forecasts)}")

# Get list of solar and wind columns for each model
print(f"\n{'='*70}")
print("AVAILABLE COLUMNS PER MODEL")
print(f"{'='*70}")

for model in forecasts:
    cols = [c for c in forecasts[model].columns if 'Forecast' in c or 'Energy' in c]
    print(f"{model:<15}: {', '.join(cols)}")

In [ ]:
print(f"\n{'='*70}")
print("METHOD 1: EXTRACT DAILY FORECASTS (All Models Side-by-Side)")
print(f"{'='*70}")

# Create unified daily forecast table
daily_output = pd.DataFrame({'Date': forecasts[list(forecasts.keys())[0]]['Date']})

# Extract solar forecasts
for model in forecasts:
    df = forecasts[model]
    # Find solar column
    solar_col = [c for c in df.columns if 'Solar' in c and 'Forecast' in c]
    if solar_col:
        daily_output[f'Solar_{model}'] = df[solar_col[0]].values
    else:
        # Try alternate column name
        other_cols = [c for c in df.columns if c != 'Date' and 'Wind' not in c]
        if other_cols:
            daily_output[f'Solar_{model}'] = df[other_cols[0]].values

# Extract wind forecasts if available
has_wind = False
for model in forecasts:
    df = forecasts[model]
    wind_col = [c for c in df.columns if 'Wind' in c and 'Forecast' in c]
    if wind_col:
        has_wind = True
        daily_output[f'Wind_{model}'] = df[wind_col[0]].values

# Calculate consensus forecasts
solar_cols = [c for c in daily_output.columns if 'Solar_' in c]
if solar_cols:
    daily_output['Solar_Mean'] = daily_output[solar_cols].mean(axis=1)
    daily_output['Solar_Median'] = daily_output[solar_cols].median(axis=1)
    daily_output['Solar_Min'] = daily_output[solar_cols].min(axis=1)
    daily_output['Solar_Max'] = daily_output[solar_cols].max(axis=1)
    daily_output['Solar_Std'] = daily_output[solar_cols].std(axis=1)

if has_wind:
    wind_cols = [c for c in daily_output.columns if 'Wind_' in c]
    if wind_cols:
        daily_output['Wind_Mean'] = daily_output[wind_cols].mean(axis=1)
        daily_output['Wind_Median'] = daily_output[wind_cols].median(axis=1)

# Save daily output
daily_output.to_csv('best_outputs_daily_2026.csv', index=False)
print(f"✓ Saved: best_outputs_daily_2026.csv")
print(f"  Shape: {daily_output.shape}")
print(f"  Columns: {daily_output.shape[1]}")
print(f"\nFirst 10 rows (Solar forecasts):")
print(daily_output[[c for c in daily_output.columns if 'Solar' in c or 'Date' in c]].head(10).to_string(index=False))

In [ ]:
print(f"\n{'='*70}")
print("METHOD 2: EXTRACT MONTHLY AGGREGATES")
print(f"{'='*70}")

# Create monthly summary
daily_output['Month'] = pd.to_datetime(daily_output['Date']).dt.to_period('M')

solar_cols_only = [c for c in daily_output.columns if 'Solar_' in c and 'Mean' not in c and 'Median' not in c and 'Min' not in c and 'Max' not in c and 'Std' not in c]

monthly_output = daily_output.groupby('Month').agg({
    **{col: ['mean', 'min', 'max', 'sum'] for col in solar_cols_only}
}).round(2)

# Flatten column names
monthly_output.columns = ['_'.join(col).strip() for col in monthly_output.columns.values]

# Save monthly output
monthly_output.to_csv('best_outputs_monthly_2026.csv')
print(f"✓ Saved: best_outputs_monthly_2026.csv")
print(f"\nMonthly Solar Energy Totals (MWh):")
print(f"\n{'Month':<12} {' '.join([f'{model:<12}' for model in forecasts.keys()])}")
print("-" * (12 + 12*len(forecasts)))

for month in monthly_output.index:
    month_str = str(month)
    row = [f"{month_str:<12}"]
    for model in forecasts:
        col = f'Solar_{model}_sum'
        val = monthly_output.loc[month, col] / 1000 if col in monthly_output.columns else 0
        row.append(f"{val:<12.1f}")
    print(''.join(row))

In [ ]:
print(f"\n{'='*70}")
print("METHOD 3: EXTRACT ANNUAL TOTALS")
print(f"{'='*70}")

annual_output = {}

for model in forecasts:
    df = forecasts[model]
    
    # Extract solar
    solar_col = [c for c in df.columns if 'Solar' in c and 'Forecast' in c]
    if solar_col:
        solar_total = df[solar_col[0]].sum() / 1000  # Convert to MWh
        solar_mean = df[solar_col[0]].mean()
        solar_std = df[solar_col[0]].std()
    else:
        other_cols = [c for c in df.columns if c != 'Date' and 'Wind' not in c]
        if other_cols:
            solar_total = df[other_cols[0]].sum() / 1000
            solar_mean = df[other_cols[0]].mean()
            solar_std = df[other_cols[0]].std()
    
    # Extract wind if available
    wind_col = [c for c in df.columns if 'Wind' in c and 'Forecast' in c]
    if wind_col:
        wind_total = df[wind_col[0]].sum() / 1000
        wind_mean = df[wind_col[0]].mean()
    else:
        wind_total = 0
        wind_mean = 0
    
    combined_total = solar_total + wind_total
    
    annual_output[model] = {
        'Solar_Total_MWh': solar_total,
        'Solar_Daily_Mean': solar_mean,
        'Solar_Daily_Std': solar_std,
        'Wind_Total_MWh': wind_total,
        'Wind_Daily_Mean': wind_mean,
        'Combined_Total_MWh': combined_total,
        'Solar_Pct': (solar_total / combined_total * 100) if combined_total > 0 else 0,
        'Wind_Pct': (wind_total / combined_total * 100) if combined_total > 0 else 0
    }

annual_df = pd.DataFrame(annual_output).T

# Save annual output
annual_df.to_csv('best_outputs_annual_2026.csv')
print(f"✓ Saved: best_outputs_annual_2026.csv")

print(f"\n{'='*70}")
print("ANNUAL TOTALS (2026)")
print(f"{'='*70}")
print(f"\n{annual_df.to_string()}")

print(f"\n\nSOLAR ANNUAL TOTALS (MWh):")
print(f"{'Model':<15} {'Total (MWh)':<15} {'Daily Mean':<15} {'Daily Std':<15}")
print("-" * 60)
for model in annual_df.index:
    print(f"{model:<15} {annual_df.loc[model, 'Solar_Total_MWh']:<15,.1f} {annual_df.loc[model, 'Solar_Daily_Mean']:<15.2f} {annual_df.loc[model, 'Solar_Daily_Std']:<15.2f}")

In [ ]:
print(f"\n{'='*70}")
print("METHOD 4: LOAD & DISPLAY VALIDATION RESULTS")
print(f"{'='*70}")

# Check if validation report exists
if Path('model_validation_report.json').exists():
    with open('model_validation_report.json', 'r') as f:
        validation = json.load(f)
    
    print(f"\n✓ Validation report found!")
    print(f"\nBest Model: {validation['best_model']}")
    print(f"Overall Score: {validation['composite_score']:.1f}/100")
    print(f"\nAccuracy Metrics:")
    for metric, value in validation['accuracy_metrics'].items():
        print(f"  {metric}: {value:.2f}")
    
    print(f"\n95% Confidence Interval:")
    print(f"  Lower: {validation['ci_95']['lower']:.2f} kWh/day")
    print(f"  Mean:  {validation['ci_95']['mean']:.2f} kWh/day")
    print(f"  Upper: {validation['ci_95']['upper']:.2f} kWh/day")
    
    print(f"\nModel Rankings:")
    for rank, (model, score) in enumerate(sorted(validation['ranking'].items(), key=lambda x: x[1], reverse=True), 1):
        print(f"  {rank}. {model:<15}: {score:.1f}/100")
    
    best_model_recommended = validation['best_model']
else:
    print(f"\n⚠ Validation report not found.")
    print(f"Run 'validate_and_select_best_model.ipynb' first to generate it.")
    best_model_recommended = 'Manual selection needed'

In [ ]:
print(f"\n{'='*70}")
print("METHOD 5: RECOMMENDED OUTPUT TO USE")
print(f"{'='*70}")

print(f"\n📊 BEST OUTPUT FOR DECISION-MAKING:")
print(f"\nFile: best_outputs_daily_2026.csv")
print(f"Contains:")
print(f"  • All 5 model daily forecasts (Solar & Wind)")
print(f"  • Consensus metrics: Mean, Median, Min, Max, Std")
print(f"  • 365 days × {daily_output.shape[1]} columns")
print(f"  • Ready for: analysis, reporting, dashboarding")

print(f"\n\n📈 FOR PRESENTATIONS:")
print(f"\nFile: best_outputs_monthly_2026.csv")
print(f"Contains:")
print(f"  • Monthly aggregates (mean, min, max, sum)")
print(f"  • 12 months × all models")
print(f"  • Better for: monthly planning, quarterly reviews")

print(f"\n\n📋 FOR EXECUTIVE SUMMARY:")
print(f"\nFile: best_outputs_annual_2026.csv")
print(f"Contains:")
print(f"  • Annual totals per model (MWh)")
print(f"  • Daily mean/std per model")
print(f"  • Solar/Wind mix percentages")
print(f"  • Perfect for: board presentations, contracts")

print(f"\n\n🏆 SINGLE BEST MODEL:")
if 'best_model_recommended' in locals() and best_model_recommended != 'Manual selection needed':
    best_file = {
        'Ensemble': 'solar_energy_forecast_2026.csv',
        'SARIMA': 'sarima_forecast_2026.csv',
        'Prophet': 'prophet_forecast_2026.csv',
        'LSTM': 'lstm_forecast_2026.csv',
        'XGBoost': 'xgboost_forecast_2026.csv'
    }.get(best_model_recommended)
    
    print(f"\nModel: {best_model_recommended}")
    print(f"File: {best_file}")
    print(f"Use case: Most accurate and stable forecasts")
    print(f"\nQuick Stats:")
    print(f"  Annual Solar: {annual_df.loc[best_model_recommended, 'Solar_Total_MWh']:,.1f} MWh")
    if annual_df.loc[best_model_recommended, 'Wind_Total_MWh'] > 0:
        print(f"  Annual Wind:  {annual_df.loc[best_model_recommended, 'Wind_Total_MWh']:,.1f} MWh")
        print(f"  Combined:     {annual_df.loc[best_model_recommended, 'Combined_Total_MWh']:,.1f} MWh")
else:
    print(f"\nRun validate_and_select_best_model.ipynb to determine best model.")
    print(f"Until then, use consensus forecasts (Mean, Median) from daily output.")

In [ ]:
print(f"\n{'='*70}")
print("METHOD 6: QUICK COMPARISON TABLE")
print(f"{'='*70}")

# Create comparison summary
comparison = pd.DataFrame({
    'Model': annual_df.index,
    'Solar_Annual_MWh': annual_df['Solar_Total_MWh'].round(1),
    'Wind_Annual_MWh': annual_df['Wind_Total_MWh'].round(1),
    'Combined_MWh': annual_df['Combined_Total_MWh'].round(1),
    'Daily_Solar_Mean': annual_df['Solar_Daily_Mean'].round(2),
    'Daily_Solar_Std': annual_df['Solar_Daily_Std'].round(2)
})

comparison = comparison.sort_values('Combined_MWh', ascending=False).reset_index(drop=True)
comparison.to_csv('best_outputs_comparison.csv', index=False)
print(f"✓ Saved: best_outputs_comparison.csv")
print(f"\n{comparison.to_string(index=False)}")

print(f"\n\nKey Insights:")
print(f"  • Highest solar forecast: {comparison.iloc[0]['Model']} ({comparison.iloc[0]['Solar_Annual_MWh']:,.1f} MWh)")
print(f"  • Lowest solar forecast:  {comparison.iloc[-1]['Model']} ({comparison.iloc[-1]['Solar_Annual_MWh']:,.1f} MWh)")
print(f"  • Range: {(comparison.iloc[0]['Solar_Annual_MWh'] - comparison.iloc[-1]['Solar_Annual_MWh']):,.1f} MWh")
print(f"  • Average: {comparison['Solar_Annual_MWh'].mean():,.1f} MWh")
print(f"  • Std Dev: {comparison['Solar_Annual_MWh'].std():,.1f} MWh")

In [ ]:
print(f"\n{'='*70}")
print("SUMMARY: OUTPUT FILES GENERATED")
print(f"{'='*70}")

output_files = [
    ('best_outputs_daily_2026.csv', 'Daily forecasts (all models)', '365 days × columns'),
    ('best_outputs_monthly_2026.csv', 'Monthly aggregates', '12 months × models'),
    ('best_outputs_annual_2026.csv', 'Annual totals & statistics', '5 models × metrics'),
    ('best_outputs_comparison.csv', 'Quick comparison table', '5 models × 6 key metrics')
]

print(f"\n{'File':<35} {'Description':<35} {'Size'}")
print("-" * 80)

for filename, desc, size in output_files:
    if Path(filename).exists():
        file_size = Path(filename).stat().st_size / 1024
        print(f"{filename:<35} {desc:<35} {file_size:>7.1f} KB")

print(f"\n{'='*70}")
print(f"Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*70}")
print(f"\n✓ All best outputs extracted successfully!")
print(f"\nNext steps:")
print(f"  1. Open best_outputs_daily_2026.csv for detailed daily forecasts")
print(f"  2. Use best_outputs_monthly_2026.csv for monthly planning")
print(f"  3. Reference best_outputs_annual_2026.csv for annual contracts")
print(f"  4. Share best_outputs_comparison.csv for stakeholder communication")